In [10]:
import json
import os
import time

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

from transformer_denoiser import TransformerPointCloudDenoiser, print_model_info
from wads_dataset_transformer import WADSDatasetTransformer
from train_transformer import LabelSmoothingCrossEntropy, EarlyStopping, WarmupCosineScheduler

In [11]:
# ───────────────────────────────
# 設定
# ───────────────────────────────
root = "./WADS/wads"
splits_file = "splits.json"
num_points = 20000  # 20000点に増加
batch_size = 4  # メモリに応じて調整
epochs = 150  # 最大エポック数を増加
lr = 2e-4  # 学習率を少し上げる
weight_decay = 1e-4

# Transformer設定
num_patches = 200  # 20000点 / 100点 = 200パッチ
embed_dim = 128
num_heads = 8
num_layers = 6
ffn_dim = 512
pool_type = 'mean'

# 学習改善設定
label_smoothing = 0.1
warmup_epochs = 10
early_stop_patience = 15

# Mixed Precision Training
use_amp = torch.cuda.is_available()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

# CUDAメモリの最適化設定
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(
        f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )
    print(f"Initial Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"Initial Reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")

# ───────────────────────────────
# Dataset / DataLoader
# ───────────────────────────────
print("\n" + "=" * 60)
print("Loading WADS Dataset (Enhanced version)...")
print("=" * 60)

train_dataset = WADSDatasetTransformer(
    root, 
    split="train", 
    num_points=num_points, 
    splits_file=splits_file,
    augment=False,  # Data augmentation ON
    rotation_range=5.0,
    scale_range=0.05,
    jitter_std=0.01
)
val_dataset = WADSDatasetTransformer(
    root, 
    split="val", 
    num_points=num_points, 
    splits_file=splits_file,
    augment=False  # Validation without augmentation
)

train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, num_workers=1, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False, num_workers=1, pin_memory=True
)

print(f"\nTrain samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

# ───────────────────────────────
# Model
# ───────────────────────────────
print("\n" + "=" * 60)
print("Initializing Enhanced Transformer Model...")
print("=" * 60)

model = TransformerPointCloudDenoiser(
    num_points=num_points,
    num_patches=num_patches,
    in_dim=4,
    embed_dim=embed_dim,
    num_heads=num_heads,
    num_layers=num_layers,
    ffn_dim=ffn_dim,
    num_classes=2,
    pool_type=pool_type,
    dropout=0.15  # Dropout増加
).to(device)

print_model_info(model)

# Loss with Label Smoothing
class_weights = torch.tensor([1.0, 10.0]).to(device)
criterion = LabelSmoothingCrossEntropy(weight=class_weights, smoothing=label_smoothing)

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

# Warmup + Cosine Annealing Scheduler
scheduler = WarmupCosineScheduler(optimizer, warmup_epochs=warmup_epochs, total_epochs=epochs)

# Early Stopping
early_stopping = EarlyStopping(patience=early_stop_patience, min_delta=0.0001, mode='max')

# Mixed Precision Scaler
scaler = torch.cuda.amp.GradScaler() if use_amp else None

Using: cuda
GPU Memory: 6.00 GB
Initial Allocated: 0.04 GB
Initial Reserved: 0.07 GB

Loading WADS Dataset (Enhanced version)...
Loading train split with sequences: [11, 12, 13, 14, 15, 16, 17, 18, 20, 22, 23, 24, 26]
  Sequence 11: 102 files
  Sequence 12: 101 files
  Sequence 13: 101 files
  Sequence 14: 101 files
  Sequence 15: 102 files
  Sequence 16: 102 files
  Sequence 17: 101 files
  Sequence 18: 101 files
  Sequence 20: 101 files
  Sequence 22: 101 files
  Sequence 23: 101 files
  Sequence 24: 101 files
  Sequence 26: 101 files
Total train samples: 1316
Loading val split with sequences: [28, 30]
  Sequence 28: 102 files
  Sequence 30: 101 files
Total val samples: 203

Train samples: 1316
Val samples: 203

Initializing Enhanced Transformer Model...

Model Information
Total parameters: 1,215,746
Trainable parameters: 1,215,746
Model size: 4.64 MB (fp32)



C:\Users\katoy\AppData\Local\Temp\ipykernel_4044\3929491148.py:110: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if use_amp else None


In [12]:
# ───────────────────────────────
# 学習開始時刻を記録
# ───────────────────────────────
start_time = time.time()
print("\n" + "=" * 60)
print("Starting Enhanced Training...")
print(f"  - Data Augmentation: ON")
print(f"  - Label Smoothing: {label_smoothing}")
print(f"  - Warmup Epochs: {warmup_epochs}")
print(f"  - Early Stopping Patience: {early_stop_patience}")
print(f"  - Mixed Precision: {'ON' if use_amp else 'OFF'}")
print("=" * 60 + "\n")

# ───────────────────────────────
# Training Loop
# ───────────────────────────────
best_val_loss = float("inf")
best_val_acc = 0.0

# 学習曲線用のリスト
train_losses = []
val_losses = []
train_accs = []
val_accs = []
learning_rates = []

for epoch in range(epochs):
    model.train()
    total_loss = 0
    total_correct = 0
    total_points = 0

    # tqdmで進捗バーを表示
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", ncols=100)

    for i, (points, labels) in enumerate(pbar):
        points = points.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        # Mixed Precision Training
        if use_amp:
            with torch.cuda.amp.autocast():
                logits = model(points)
                logits = logits.permute(0, 2, 1)
                loss = criterion(logits, labels)
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(points)
            logits = logits.permute(0, 2, 1)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        total_loss += loss.item()

        # Accuracy
        preds = logits.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_points += labels.numel()

        current_loss = total_loss / (i + 1)
        current_acc = total_correct / total_points
        pbar.set_postfix(
            {"loss": f"{current_loss:.4f}", "acc": f"{current_acc:.4f}"}
        )

        del points, labels, logits, preds, loss
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    avg_train_loss = total_loss / len(train_loader)
    train_acc = total_correct / total_points
    train_losses.append(avg_train_loss)
    train_accs.append(train_acc)
    
    current_lr = scheduler.step()
    learning_rates.append(current_lr)
    
    print(
        f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | LR: {current_lr:.6f}"
    )

    # ───────────────────────────────
    # Validation
    # ───────────────────────────────
    model.eval()
    val_loss = 0
    correct = 0
    total = 0

    class_correct = [0, 0]
    class_total = [0, 0]

    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]  ", ncols=100)

    with torch.no_grad():
        for points, labels in val_pbar:
            points = points.to(device)
            labels = labels.to(device)

            if use_amp:
                with torch.cuda.amp.autocast():
                    logits = model(points)
                    logits_perm = logits.permute(0, 2, 1)
                    loss = criterion(logits_perm, labels)
            else:
                logits = model(points)
                logits_perm = logits.permute(0, 2, 1)
                loss = criterion(logits_perm, labels)
                
            val_loss += loss.item()

            preds = logits_perm.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.numel()

            for c in range(2):
                class_mask = labels == c
                class_correct[c] += ((preds == labels) & class_mask).sum().item()
                class_total[c] += class_mask.sum().item()

            del points, labels, logits, logits_perm, preds, loss
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    avg_val_loss = val_loss / len(val_loader)
    val_acc = correct / total
    val_losses.append(avg_val_loss)
    val_accs.append(val_acc)

    class0_acc = class_correct[0] / class_total[0] if class_total[0] > 0 else 0
    class1_acc = class_correct[1] / class_total[1] if class_total[1] > 0 else 0

    print(f"  Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(
        f"  Class 0 (Normal) Acc: {class0_acc:.4f} | Class 1 (Noise) Acc: {class1_acc:.4f}"
    )

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_val_loss = avg_val_loss
        os.makedirs("checkpoints", exist_ok=True)
        save_path = "checkpoints/transformer_best.pth"
        torch.save(model.state_dict(), save_path)
        print(f"  ★ Best model saved: {save_path} (Val Acc: {val_acc:.4f})")
    
    # Early Stopping Check
    if early_stopping(val_acc):
        print(f"\n⚠ Early stopping triggered at epoch {epoch+1}")
        print(f"  Best validation accuracy: {best_val_acc:.4f}")
        break

# ───────────────────────────────
# Save final model
# ───────────────────────────────
os.makedirs("checkpoints", exist_ok=True)
save_path = "checkpoints/transformer_final.pth"
torch.save(model.state_dict(), save_path)
print(f"\nFinal model saved to {save_path}")

# ───────────────────────────────
# 学習曲線データをJSONファイルに保存
# ───────────────────────────────
os.makedirs("results", exist_ok=True)
metrics_data = {
    "train_losses": train_losses,
    "val_losses": val_losses,
    "train_accs": train_accs,
    "val_accs": val_accs,
    "learning_rates": learning_rates,
    "epochs_trained": len(train_losses),
    "max_epochs": epochs,
    "batch_size": batch_size,
    "learning_rate": lr,
    "num_points": num_points,
    "num_patches": num_patches,
    "embed_dim": embed_dim,
    "num_heads": num_heads,
    "num_layers": num_layers,
    "ffn_dim": ffn_dim,
    "label_smoothing": label_smoothing,
    "warmup_epochs": warmup_epochs,
    "early_stop_patience": early_stop_patience,
    "dataset": "WADS",
    "model": "Transformer_Enhanced"
}

json_path = "results/transformer_training_metrics.json"
with open(json_path, "w") as f:
    json.dump(metrics_data, f, indent=4)
print(f"Training metrics saved to {json_path}")

# ───────────────────────────────
# 学習曲線の描画
# ───────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Loss curve
axes[0, 0].plot(
    range(1, len(train_losses) + 1), train_losses, label="Train Loss", marker="o", linewidth=2
)
axes[0, 0].plot(
    range(1, len(val_losses) + 1),
    val_losses,
    label="Validation Loss",
    marker="s",
    linewidth=2,
)
axes[0, 0].set_xlabel("Epoch", fontsize=12)
axes[0, 0].set_ylabel("Loss", fontsize=12)
axes[0, 0].set_title("Training and Validation Loss", fontsize=14)
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# Accuracy curve
axes[0, 1].plot(
    range(1, len(train_accs) + 1), train_accs, label="Train Acc", marker="o", linewidth=2
)
axes[0, 1].plot(
    range(1, len(val_accs) + 1),
    val_accs,
    label="Validation Acc",
    marker="s",
    linewidth=2,
)
axes[0, 1].set_xlabel("Epoch", fontsize=12)
axes[0, 1].set_ylabel("Accuracy", fontsize=12)
axes[0, 1].set_title("Training and Validation Accuracy", fontsize=14)
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

# Learning rate curve
axes[1, 0].plot(
    range(1, len(learning_rates) + 1), learning_rates, label="Learning Rate", marker="o", linewidth=2, color='green'
)
axes[1, 0].set_xlabel("Epoch", fontsize=12)
axes[1, 0].set_ylabel("Learning Rate", fontsize=12)
axes[1, 0].set_title("Learning Rate Schedule", fontsize=14)
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(True, alpha=0.3)

# Summary stats
axes[1, 1].axis('off')
summary_text = f"""
Training Summary
─────────────────────────────
Epochs Trained: {len(train_losses)} / {epochs}
Best Val Accuracy: {best_val_acc:.4f}
Best Val Loss: {best_val_loss:.4f}
Final Train Acc: {train_accs[-1]:.4f}
Final Val Acc: {val_accs[-1]:.4f}

Model Configuration:
─────────────────────────────
Points: {num_points}
Patches: {num_patches}
Embed Dim: {embed_dim}
Layers: {num_layers}
Heads: {num_heads}

Training Configuration:
─────────────────────────────
Data Augmentation: ON
Label Smoothing: {label_smoothing}
Warmup Epochs: {warmup_epochs}
Early Stopping: {early_stop_patience} epochs
"""
axes[1, 1].text(0.1, 0.5, summary_text, fontsize=10, verticalalignment='center', family='monospace')

plt.tight_layout()

curve_path = "results/transformer_learning_curve.png"
plt.savefig(curve_path, dpi=150)
print(f"Learning curve saved to {curve_path}")
plt.close()

# ───────────────────────────────
# トータル学習時間を表示
# ───────────────────────────────
end_time = time.time()
total_time = end_time - start_time

hours = int(total_time // 3600)
minutes = int((total_time % 3600) // 60)
seconds = int(total_time % 60)

print("\n" + "=" * 60)
print(f"Training completed!")
print(
    f"Total training time: {hours:02d}:{minutes:02d}:{seconds:02d} ({total_time:.2f} seconds)"
)
print(f"Best validation accuracy: {best_val_acc:.4f}")
print(f"Best validation loss: {best_val_loss:.4f}")
print("=" * 60)


Starting Enhanced Training...
  - Data Augmentation: ON
  - Label Smoothing: 0.1
  - Warmup Epochs: 10
  - Early Stopping Patience: 15
  - Mixed Precision: ON



Epoch 1/150 [Train]:   0%|                                                  | 0/329 [00:00<?, ?it/s]C:\Users\katoy\AppData\Local\Temp\ipykernel_4044\3785469032.py:44: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/150 [Train]: 100%|███████████████| 329/329 [00:22<00:00, 14.51it/s, loss=0.6510, acc=0.6705]


Epoch 1 | Train Loss: 0.6510 | Train Acc: 0.6705 | LR: 0.000020


Epoch 1/150 [Val]  :   0%|                                                   | 0/51 [00:00<?, ?it/s]C:\Users\katoy\AppData\Local\Temp\ipykernel_4044\3785469032.py:111: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/150 [Val]  : 100%|██████████████████████████████████████████| 51/51 [00:05<00:00,  9.18it/s]


  Val Loss: 0.5081 | Val Acc: 0.9078
  Class 0 (Normal) Acc: 1.0000 | Class 1 (Noise) Acc: 0.0000
  ★ Best model saved: checkpoints/transformer_best.pth (Val Acc: 0.9078)


...

Epoch 3 | Train Loss: 0.5544 | Train Acc: 0.8378 | LR: 0.000060


Epoch 3/150 [Val]  : 100%|██████████████████████████████████████████| 51/51 [00:05<00:00, 10.01it/s]


  Val Loss: 0.4907 | Val Acc: 0.9075
  Class 0 (Normal) Acc: 1.0000 | Class 1 (Noise) Acc: 0.0000


Epoch 4/150 [Train]:  13%|██              | 43/329 [00:04<00:29,  9.68it/s, loss=0.5390, acc=0.8566]


KeyboardInterrupt: 